# READ ME
# Fireworks Crowd Dashboard

An AI-powered dashboard that recommends the least crowded fireworks viewing spot based on crowd ranking, distance, and ETA.

Technologies:
- Python
- Streamlit
- Folium
- Geopy
- YOLOv8


In [14]:
import os

os.makedirs("app", exist_ok=True)
os.makedirs("output", exist_ok=True)

**Step 2: read the JSON**

In [15]:
import os
os.listdir("output")

['ranking_output.json']

In [16]:
import json

In [17]:
with open("output/ranking_output.json", "r") as f:
    data = json.load(f)

print(data)

[{'rank': 1, 'spot_id': 'spot_3', 'spot_name': 'Spot 3', 'head_count': 53, 'density': None, 'crowd_level': 'Moderate'}, {'rank': 2, 'spot_id': 'spot_2', 'spot_name': 'Spot 2', 'head_count': 79, 'density': None, 'crowd_level': 'High'}, {'rank': 3, 'spot_id': 'spot_1', 'spot_name': 'Spot 1', 'head_count': 99, 'density': None, 'crowd_level': 'High'}]


In [18]:
import pandas as pd

df = pd.DataFrame(data)
df.head()

,rank,spot_id,spot_name,head_count,density,crowd_level
0,1,spot_3,Spot 3,53,None,Moderate
1,2,spot_2,Spot 2,79,None,High
2,3,spot_1,Spot 1,99,None,High


**Step 3: Fake coordinates add karo**

In [19]:
spot_coordinates = {
    "spot_1": (24.8607, 67.0011),
    "spot_2": (24.8650, 67.0050),
    "spot_3": (24.8700, 67.0100)
}

**Step 4: User location define karo**

In [20]:
user_location = (24.8550, 67.0300)

**Step 5: Distance calculate karot**

In [21]:
!pip install geopy

In [22]:
from geopy.distance import geodesic

distances = []

for spot_id, coords in spot_coordinates.items():
    distance = geodesic(user_location, coords).km
    distances.append((spot_id, distance))

print(distances)

[('spot_1', 2.988270580588746), ('spot_2', 2.758758287928575), ('spot_3', 2.6165269744239272)]


**Step 6: Distance ko dataframe mein add karo**

In [23]:
df["distance_km"] = df["spot_id"].map(
    lambda x: geodesic(
        user_location,
        spot_coordinates[x]
    ).km
)

**Step 7: ETA calculate karo**

Assume walking speed = 5 km/h.

In [24]:
df["eta_minutes"] = (df["distance_km"] / 5) * 60

**Step 8: Best spot choose karo**

Least crowded aur nearest spot.

In [25]:
best_spot = df.sort_values(
    ["rank", "distance_km"]
).iloc[0]

print(best_spot)

rank                   1
spot_id           spot_3
spot_name         Spot 3
head_count            53
density             None
crowd_level     Moderate
distance_km     2.616527
eta_minutes    31.398324
Name: 0, dtype: object


**Step 9: Map banao**

In [26]:
!pip install folium

In [27]:
import folium

m = folium.Map(location=user_location, zoom_start=14)

folium.Marker(
    user_location,
    popup="You"
).add_to(m)

for spot, coords in spot_coordinates.items():
    folium.Marker(
        coords,
        popup=spot
    ).add_to(m)

m

**Step 10: Streamlit app**

In [28]:
!pip install streamlit

Basic app:

In [29]:
import streamlit as st

st.title("Fireworks Crowd Dashboard")

st.dataframe(df)

st.write("Best Spot:")
st.write(best_spot)

2026-07-10 17:57:37.319 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-10 17:57:37.323 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-10 17:57:37.323 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-10 17:57:37.369 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-10 17:57:37.371 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-10 17:57:37.372 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-10 17:57:37.374 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-10 17:57:37.377 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

**Step 11: Functions banao**

In [30]:
import json
import pandas as pd
from geopy.distance import geodesic

def load_data(file_path):
    with open(file_path, "r") as f:
        data = json.load(f)
    return pd.DataFrame(data)


def add_distance_eta(df, user_location, spot_coordinates):
    df["distance_km"] = df["spot_id"].map(
        lambda x: geodesic(
            user_location,
            spot_coordinates[x]
        ).km
    )

    df["eta_minutes"] = (
        df["distance_km"] / 5
    ) * 60

    return df

**Step 12: Best spot function**

In [31]:
def get_best_spot(df):
    best_spot = df.sort_values(
        ["rank", "distance_km"]
    ).iloc[0]

    return best_spot

**Step 13: Create app.py**

In [35]:
import streamlit as st
import pandas as pd
import json
from geopy.distance import geodesic
import folium
!pip install streamlit_folium
from streamlit_folium import st_folium

st.title("🎆 Fireworks Crowd Dashboard")

with open("output/ranking_output.json", "r") as f:
    data = json.load(f)

df = pd.DataFrame(data)

# Explicitly convert object columns to string type to prevent PyArrow errors
# The error mentions 'spot_3' being converted to int64, so spot_id is a candidate.
# The 'density' column also has None, which can cause issues.
df['spot_id'] = df['spot_id'].astype(str)
df['density'] = df['density'].astype(str)

spot_coordinates = {
    "spot_1": (24.8607, 67.0011),
    "spot_2": (24.8650, 67.0050),
    "spot_3": (24.8700, 67.0100)
}

user_location = (24.8550, 67.0300)

df["distance_km"] = df["spot_id"].map(
    lambda x: geodesic(
        user_location,
        spot_coordinates[x]
    ).km
)

df["eta_minutes"] = (
    df["distance_km"] / 5
) * 60

best_spot = df.sort_values(
    ["rank", "distance_km"]
).iloc[0]

st.subheader("Crowd Ranking")
st.dataframe(df)

st.subheader("Best Spot")
st.write(best_spot)

2026-07-10 18:02:22.989 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-10 18:02:22.989 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-10 18:02:22.991 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-10 18:02:22.999 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-10 18:02:23.000 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-10 18:02:23.001 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-10 18:02:23.003 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-10 18:02:23.004 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bar

**Step 14: Add map**

In [36]:
m = folium.Map(
    location=user_location,
    zoom_start=14
)

folium.Marker(
    user_location,
    popup="You"
).add_to(m)

for spot, coords in spot_coordinates.items():
    folium.Marker(
        coords,
        popup=spot
    ).add_to(m)

st_folium(m)

2026-07-10 18:03:15.231 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-10 18:03:15.232 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-10 18:03:15.234 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-10 18:03:15.234 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.
2026-07-10 18:03:15.235 Thread 'MainThread': missing ScriptRunContext! This warning can be ignored when running in bare mode.


{'last_clicked': None,
 'last_object_clicked': None,
 'last_object_clicked_tooltip': None,
 'last_object_clicked_popup': None,
 'all_drawings': None,
 'last_active_drawing': None,
 'bounds': {'_southWest': {'lat': 24.855, 'lng': 67.0011},
  '_northEast': {'lat': 24.87, 'lng': 67.03}},
 'zoom': 14,
 'last_circle_radius': None,
 'last_circle_polygon': None,
 'selected_layers': None,
 'selected_tags': None,
 'last_geocoder_result': None}

**Step 15: Create requirements.txt**

In [37]:
%%writefile requirements.txt
streamlit
pandas
geopy
folium
streamlit-folium

Writing requirements.txt


In [38]:
!cat requirements.txt

streamlit
pandas
geopy
folium
streamlit-folium


In [44]:
!zip -r fireworks_project.zip app.py requirements.txt output *.ipynb

	zip warning: name not matched: app.py
	zip warning: name not matched: *.ipynb
  adding: requirements.txt (deflated 21%)
  adding: output/ (stored 0%)
  adding: output/ranking_output.json (deflated 67%)


In [45]:
from google.colab import files
files.download("fireworks_project.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>